# Module 6: End-to-End Orchestration (Vertex AI Pipelines)

This notebook orchestrates the entire lifecycle we've built—Data Generation, Export, Fine-Tuning, Model Registry, and Deployment—into a single **Vertex AI Pipeline** (based on Kubeflow Pipelines). 

**Lineage Tracking**: This pipeline is designed using native `kfp.dsl.Dataset` and `kfp.dsl.Model` artifacts. By passing these artifacts between `@dsl.component` functions, Vertex AI cleanly maps the entire execution lineage within the ML Metadata graph.


In [ ]:
.pip install kfp google-cloud-aiplatform google-cloud-pipeline-components python-dotenv -q
# Ensure you restart the kernel after installing KFP if running interactively.


In [ ]:
import os
from google.cloud import aiplatform
from kfp import dsl, compiler
from kfp.dsl import Dataset, Model, Input, Output
from dotenv import load_dotenv

load_dotenv()

PROJECT_ID = os.getenv("PROJECT_ID", "YOUR_PROJECT_ID")
LOCATION = os.getenv("LOCATION", "global") 
STAGING_BUCKET = os.getenv("STAGING_BUCKET", "YOUR_GCS_BUCKET_NAME")
DATASET_ID = os.getenv("DATASET_ID", "tuning")

if not STAGING_BUCKET.startswith("gs://"):
    STAGING_BUCKET = f"gs://{STAGING_BUCKET}"

aiplatform.init(project=PROJECT_ID, location="us-central1", staging_bucket=STAGING_BUCKET)
print(f"Project ID: {PROJECT_ID}")


## Step 1: Data Preparation in BigQuery Component

This serverless component executes the raw BigQuery ML generating SQL natively to load 1000 production scenarios.


In [ ]:
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-bigquery", "google-cloud-storage"]
)
def run_bq_data_generation(project_id: str, location: str, dataset_id: str):
    from google.cloud import bigquery
    client = bigquery.Client(project=project_id, location="US")
    
    # Executing the full generative SQL pipeline
    query_generate_data = r"""
-- generate_synthetic_data.sql
-- BigQuery ML Generator for Data Gravity Demo
-- 1. Setup DDL: Create Connection
-- We use a dedicated connection in the 'us-central1' region.
-- IMPORTANT: The service account created by this connection needs 'roles/aiplatform.user'.
-- 2. Setup DDL: Create Model
CREATE OR REPLACE MODEL `tuning.gemini_simulator`
REMOTE WITH CONNECTION `us.vertex_ai_conn_us`
OPTIONS (
  endpoint = 'projects/data-n-models/locations/global/publishers/google/models/gemini-3-flash-preview'
);

-- 2. Create Downstream Tables
CREATE OR REPLACE TABLE `tuning.cpt_data` (
  id STRING,
  category STRING,
  theme STRING,
  initial_description STRING,
  raw_artifact STRING,
  investigation_notes STRING,
  remediation_steps STRING,
  q_and_a_pairs JSON
);

CREATE OR REPLACE TABLE `tuning.sft_data` (
  cpt_id STRING,
  messages JSON
);

CREATE OR REPLACE TABLE `tuning.lora_data` (
  cpt_id STRING,
  messages JSON
);

-- Uncomment to reset the entire dataset completely:
TRUNCATE TABLE `tuning.cpt_data`;

-- ==============================================================================
-- STEP 0: SEED CONTEXTS (NATIVE BIGQUERY ML)
-- Generates abstract UUID scenarios natively without Python.
-- ==============================================================================

-- 0.1: Instantiate 1,000 empty scenarios (Test Scale)
INSERT INTO `tuning.cpt_data` (id, category)
SELECT
  GENERATE_UUID() AS id,
  ['SOC Telemetry / SIEM Logs', 'JIRA/ServiceNow Analyst Investigation Ticket', 'Application Security Bug Fix (Git Diff and developer comments)', 'Threat Intelligence Actor Profile or Post-Mortem', 'Cloud Architecture Threat Model or IaC Misconfiguration'][OFFSET(CAST(FLOOR(RAND() * 5) AS INT64))] AS category
FROM UNNEST(GENERATE_ARRAY(1, 1000))
WHERE NOT EXISTS (SELECT 1 FROM `tuning.cpt_data` LIMIT 1);

-- 0.2: Generate Themes and Descriptions via Gemini
CREATE TEMP TABLE temp_themes AS
WITH prompt_data AS (
  SELECT
    id,
    CONCAT(
      'Generate exactly one highly technical, unique cybersecurity scenario. ',
      'Focus specifically on the reporting category: ', category, '. ',
      'This scenario will be used as a seed to generate detailed training artifacts later. ',
      'Output strictly a JSON object with two keys: "theme" (2-5 words) and "initial_description" (2-3 sentences outlining the scenario context). Do not output markdown or any other text.'
    ) AS prompt
  FROM `tuning.cpt_data`
  WHERE theme IS NULL
)
SELECT
  id,
  SAFE.PARSE_JSON(
    REPLACE(REPLACE(JSON_EXTRACT_SCALAR(ml_generate_text_result, '$.candidates[0].content.parts[0].text'), '```json', ''), '```', '')
  ) AS generated_json
FROM ML.GENERATE_TEXT(
  MODEL `tuning.gemini_simulator`,
  TABLE prompt_data,
  STRUCT(
    1.0 AS temperature,
    65536 AS max_output_tokens,
    0.95 AS top_p,
    40 AS top_k
  )
);

UPDATE `tuning.cpt_data` AS t
SET 
  theme = CAST(JSON_EXTRACT_SCALAR(temp.generated_json, '$.theme') AS STRING),
  initial_description = CAST(JSON_EXTRACT_SCALAR(temp.generated_json, '$.initial_description') AS STRING)
FROM temp_themes AS temp
WHERE t.id = temp.id;


-- ==============================================================================
-- STEP 1: GENERATE RAW CPT ARTIFACTS
-- Updates the `cpt_data` master table dynamically based on category.
-- ==============================================================================
CREATE TEMP TABLE temp_artifacts AS
WITH prompt_data AS (
  SELECT
    id,
    CONCAT(
      'Act as an expert Cybersecurity Instructor and Red Team operator. ',
      'I am building a training dataset. Analyze this scenario:\n',
      'Category: ', category, '\n',
      'Theme: ', theme, '\n',
      'Description: ', initial_description, '\n\n',
      'Based STRICTLY on the Category above, generate a highly dense, realistic, and highly technical artifact representing this scenario. ',
      'If the category is SIEM/Telemetry, output raw log dumps. ',
      'If the category is JIRA/ServiceNow, output the raw JSON of an analyst ticket and investigation notes. ',
      'If the category is AppSec, output a Git Diff with developer comments explaining the vulnerability and patch. ',
      'If the category is Threat Intel, output a raw Markdown intelligence report. ',
      'If the category is Cloud Architecture, output a vulnerable IAM JSON policy or Terraform snippet. ',
      'Do NOT include any introductory or concluding text. Output ONLY the raw artifact data.'
    ) AS prompt
  FROM `tuning.cpt_data`
  WHERE raw_artifact IS NULL AND theme IS NOT NULL
)
SELECT
  id,
  JSON_EXTRACT_SCALAR(ml_generate_text_result, '$.candidates[0].content.parts[0].text') AS generated_artifact
FROM ML.GENERATE_TEXT(
  MODEL `tuning.gemini_simulator`,
  TABLE prompt_data,
  STRUCT(
    1.0 AS temperature,
    65536 AS max_output_tokens,
    0.95 AS top_p,
    40 AS top_k
  )
);

UPDATE `tuning.cpt_data` AS t
SET raw_artifact = temp.generated_artifact
FROM temp_artifacts AS temp
WHERE t.id = temp.id;


-- ==============================================================================
-- STEP 2: GENERATE QA DATA (SFT/LoRA Staging in master table)
-- Updates the `cpt_data` master table with structured QA based on the artifacts.
-- ==============================================================================
CREATE TEMP TABLE temp_qa_pairs AS
WITH prompt_data AS (
  SELECT
    id,
    CONCAT(
      'Act as an expert SOC Analyst and AppSec instructor. ',
      'Read this raw cybersecurity artifact data:\n', raw_artifact, '\n\n',
      'Your task is to output a STRICT JSON array containing exactly 3 highly technical Interaction pairs based STRICTLY on the artifact data above. ',
      'The interactions should emulate a real analyst conversation with an AI assistant. Ensure realistic variation: ',
      '- Interaction 1 should be a simple, direct question asking for identification or an overview of a specific indicator. ',
      '- Interaction 2 should feature the analyst explicitly copy-pasting a chunk of the raw artifact data as context inside their question, asking a targeted question about that specific snippet. ',
      '- Interaction 3 should be a very complex, multi-part analytical question requiring deep reasoning across multiple parts of the artifact. ',
      'The answers must be helpful, highly technical, and accurate to the artifact.\n\n',
      'Output format MUST be strictly a JSON array of objects:\n',
      '[{"question": "..", "answer": ".."}, {"question": "..", "answer": ".."}, {"question": "..", "answer": ".."}]'
    ) AS prompt
  FROM `tuning.cpt_data`
  WHERE q_and_a_pairs IS NULL AND raw_artifact IS NOT NULL
)
SELECT
  id,
  SAFE.PARSE_JSON(
    REPLACE(REPLACE(JSON_EXTRACT_SCALAR(ml_generate_text_result, '$.candidates[0].content.parts[0].text'), '```json', ''), '```', '')
  ) AS generated_qa
FROM ML.GENERATE_TEXT(
  MODEL `tuning.gemini_simulator`,
  TABLE prompt_data,
  STRUCT(
    1.0 AS temperature,
    65536 AS max_output_tokens,
    0.95 AS top_p,
    40 AS top_k
  )
);

UPDATE `tuning.cpt_data` AS t
SET q_and_a_pairs = temp.generated_qa
FROM temp_qa_pairs AS temp
WHERE t.id = temp.id;

-- ==============================================================================
-- STEP 3: PARSE AND POPULATE SFT DATA
-- SFT requires Vertex AI conversational formatting (messages array)
-- ==============================================================================
TRUNCATE TABLE `tuning.sft_data`;

INSERT INTO `tuning.sft_data` (cpt_id, messages)
SELECT
  id AS cpt_id,
  JSON_ARRAY(
      JSON_OBJECT('role', 'system', 'content', 'You are an expert AI security analyst assistant.'),
      JSON_OBJECT('role', 'user', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[0].question') AS STRING)),
      JSON_OBJECT('role', 'assistant', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[0].answer') AS STRING))
  ) AS messages
FROM `tuning.cpt_data`
WHERE JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[0].question') IS NOT NULL
UNION ALL
SELECT
  id AS cpt_id,
  JSON_ARRAY(
      JSON_OBJECT('role', 'system', 'content', 'You are an expert AI security analyst assistant.'),
      JSON_OBJECT('role', 'user', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[1].question') AS STRING)),
      JSON_OBJECT('role', 'assistant', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[1].answer') AS STRING))
  ) AS messages
FROM `tuning.cpt_data`
WHERE JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[1].question') IS NOT NULL
UNION ALL
SELECT
  id AS cpt_id,
  JSON_ARRAY(
      JSON_OBJECT('role', 'system', 'content', 'You are an expert AI security analyst assistant.'),
      JSON_OBJECT('role', 'user', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[2].question') AS STRING)),
      JSON_OBJECT('role', 'assistant', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[2].answer') AS STRING))
  ) AS messages
FROM `tuning.cpt_data`
WHERE JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[2].question') IS NOT NULL;

-- ==============================================================================
-- STEP 4: PARSE AND POPULATE LORA DATA
-- LoRA uses identical formatting to SFT based on specification
-- ==============================================================================
TRUNCATE TABLE `tuning.lora_data`;

INSERT INTO `tuning.lora_data` (cpt_id, messages)
SELECT
  id AS cpt_id,
  JSON_ARRAY(
      JSON_OBJECT('role', 'system', 'content', 'You are an expert AI security analyst assistant.'),
      JSON_OBJECT('role', 'user', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[0].question') AS STRING)),
      JSON_OBJECT('role', 'assistant', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[0].answer') AS STRING))
  ) AS messages
FROM `tuning.cpt_data`
WHERE JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[0].question') IS NOT NULL
UNION ALL
SELECT
  id AS cpt_id,
  JSON_ARRAY(
      JSON_OBJECT('role', 'system', 'content', 'You are an expert AI security analyst assistant.'),
      JSON_OBJECT('role', 'user', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[1].question') AS STRING)),
      JSON_OBJECT('role', 'assistant', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[1].answer') AS STRING))
  ) AS messages
FROM `tuning.cpt_data`
WHERE JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[1].question') IS NOT NULL
UNION ALL
SELECT
  id AS cpt_id,
  JSON_ARRAY(
      JSON_OBJECT('role', 'system', 'content', 'You are an expert AI security analyst assistant.'),
      JSON_OBJECT('role', 'user', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[2].question') AS STRING)),
      JSON_OBJECT('role', 'assistant', 'content', CAST(JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[2].answer') AS STRING))
  ) AS messages
FROM `tuning.cpt_data`
WHERE JSON_EXTRACT_SCALAR(q_and_a_pairs, '$[2].question') IS NOT NULL;

    """
    
    job = client.query(query_generate_data)
    job.result()
    print("Successfully triggered Native BigQuery Data Generation.")


In [ ]:
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-bigquery", "google-cloud-storage"]
)
def export_bq_to_gcs(project_id: str, location: str, dataset_id: str, staging_bucket: str, dataset_out: Output[Dataset]):
    from google.cloud import bigquery
    import time
    client = bigquery.Client(project=project_id, location="US")
    
    staging_path = staging_bucket.replace("gs://", "")
    bucket_name = staging_path.split("/")[0]
    base_prefix = staging_path[len(bucket_name)+1:] if "/" in staging_path else ""
    
    timestamp = str(int(time.time()))
    staging_clean = staging_bucket.rstrip("/")
    train_uri = f"{staging_clean}/tuning_data/pipeline_sft_train_{timestamp}_*.jsonl"
    
    query = f"""
    EXPORT DATA OPTIONS(
      uri='{train_uri}',
      format='JSON',
      overwrite=true
    ) AS
    SELECT messages 
    FROM `{project_id}.{dataset_id}.sft_data`
    WHERE messages IS NOT NULL AND RAND() < 0.9 LIMIT 1000;
    """
    job = client.query(query)
    job.result()
    
    from google.cloud import storage
    storage_client = storage.Client(project=project_id)
    bucket_obj = storage_client.bucket(bucket_name)
    
    prefix = f"{base_prefix}/tuning_data/pipeline_sft_train_{timestamp}_" if base_prefix else f"tuning_data/pipeline_sft_train_{timestamp}_"
    blobs = list(bucket_obj.list_blobs(prefix=prefix))
    exact_uri = f"gs://{bucket_name}/{blobs[0].name}"
    
    # Register lineage URI to track in Vertex ML Metadata
    dataset_out.uri = exact_uri
    dataset_out.metadata["dataset_id"] = dataset_id
    print(f"Successfully exported dataset to {exact_uri}")


In [ ]:
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-aiplatform"]
)
def run_sft_tuning(project_id: str, dataset_in: Input[Dataset], base_model: str, epochs: int, staging_bucket: str, model_out: Output[Model]):
    from vertexai.tuning import sft
    import vertexai
    import time
    
    vertexai.init(project=project_id, location="us-central1")
    
    timestamp = str(int(time.time()))
    staging_clean = staging_bucket.rstrip("/")
    output_dir = f"{staging_clean}/tuning_output/pipeline_{timestamp}"
    
    print(f"Starting Supervised Tuning Job on {base_model}..")
    sft_tuning_job = sft.train(
        source_model=base_model,
        train_dataset=dataset_in.uri,
        epochs=epochs,
        output_uri=output_dir
    )
    
    tuned_model_name = sft_tuning_job.tuned_model_name
    print(f"Model tuned successfully: {tuned_model_name}")
    
    # Store Vertex AI Model Resource Name natively into the KFP pipeline Model artifact
    model_out.uri = f"https://us-central1-aiplatform.googleapis.com/v1/{tuned_model_name}"
    model_out.metadata["resourceName"] = tuned_model_name


In [ ]:
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-aiplatform"]
)
def deploy_tuned_model(project_id: str, model_in: Input[Model], endpoint_name: str):
    from google.cloud import aiplatform
    aiplatform.init(project=project_id, location="us-central1")
    
    tuned_model_id = model_in.metadata.get("resourceName", "")
    
    if not tuned_model_id:
        print("Tuning job produced raw weights in GCS. Deployment of Open Models (e.g., Llama) requires a custom inference container step instead of auto-deployment. Pipeline SFT step finished successfully.")
        return

    model = aiplatform.Model(tuned_model_id)
    
    # Model Registry
    try:
        model.version_aliases = ['production']
        print("Alias 'production' set in Model Registry.")
    except Exception as e:
        print(f"Could not set alias: {e}")
        
    # Endpoint Deployment
    endpoints = aiplatform.Endpoint.list(filter=f'display_name="{endpoint_name}"')
    if endpoints:
        endpoint = endpoints[0]
        print(f"Found existing endpoint: {endpoint.name}")
    else:
        endpoint = aiplatform.Endpoint.create(display_name=endpoint_name)
        print(f"Created endpoint: {endpoint.name}")
        
    print("Deploying model (Async) to avoid blocking the pipeline worker..")
    model.deploy(
        endpoint=endpoint,
        machine_type="g2-standard-12",
        accelerator_type="NVIDIA_L4",
        accelerator_count=1,
        sync=False 
    )


## Define and Compile the Pipeline DAG
Using `Input[]` and `Output[]` ensures standard nodes are visually tracked within the UI.


In [ ]:
@dsl.pipeline(
    name="cyber-tuning-e2e-pipeline",
    description="End-to-end Kubeflow pipeline for data gen, tuning, and deployment with 1,000 scenario tests."
)
def cyber_tuning_pipeline(
    project_id: str,
    location: str,
    dataset_id: str,
    staging_bucket: str,
    base_model: str = "meta/llama3_1@llama-3.1-8b",
    endpoint_name: str = "pipeline-cyber-endpoint",
    epochs: int = 1
):
    # 1: Generate Data
    data_gen_task = run_bq_data_generation(
        project_id=project_id,
        location=location,
        dataset_id=dataset_id
    ).set_display_name("1. BigQuery Native Data Generation (1K)")
    
    # 2: Export to GCS
    export_task = export_bq_to_gcs(
        project_id=project_id,
        location=location,
        dataset_id=dataset_id,
        staging_bucket=staging_bucket
    ).set_display_name("2. Native Export to GCS").after(data_gen_task)
    
    # 3: Vertex SFT Tuning
    tuning_task = run_sft_tuning(
        project_id=project_id,
        dataset_in=export_task.outputs["dataset_out"],
        base_model=base_model,
        epochs=epochs,
        staging_bucket=staging_bucket
    ).set_display_name("3. SFT Fine-Tuning").set_caching_options(True)
    
    # 4 & 5: Model Registry and Endpoint Deployment
    deploy_task = deploy_tuned_model(
        project_id=project_id,
        model_in=tuning_task.outputs["model_out"],
        endpoint_name=endpoint_name
    ).set_display_name("4 & 5. Model Registry & App Deployment")

compiler.Compiler().compile(
    pipeline_func=cyber_tuning_pipeline,
    package_path="cyber_tuning_pipeline.json"
)
print("Pipeline compiled successfully to cyber_tuning_pipeline.json")


## Submit the Pipeline to Vertex AI Pipelines Engine


In [ ]:
import datetime
pipeline_job = aiplatform.PipelineJob(
    display_name=f"cyber-tuning-e2e-run-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}",
    template_path="cyber_tuning_pipeline.json",
    pipeline_root=f"{STAGING_BUCKET}/pipeline_root",
    parameter_values={
        "project_id": PROJECT_ID,
        "location": LOCATION,
        "dataset_id": DATASET_ID,
        "staging_bucket": STAGING_BUCKET,
    },
    location="us-central1"
)
pipeline_job.submit()
print(f"Pipeline submitted to Vertex AI. View it natively in the Cloud Console.")
